# Phase 4 - Pose Chaining + Depth Backprojection -> Open3D Point Cloud

Turns Phase 3's trained Mini-3D-Recon predictions into a fused 3D point
cloud. Camera intrinsics/depth-units for UnityCam were never confirmed
anywhere -- a sourced-but-unconfirmed candidate (FOV=91.32 deg, near=0.01,
far=2, from the actual Unity project's Record_scene.unity) is used, gated
by an empirical visual check: GT-mode reconstruction (this dataset's own
depth+pose, no model) must look like a coherent stomach-lumen tube before
predicted-mode output is trusted as a real Phase 3 quality signal. See
PROGRESS.md "Phase 4 camera model -- sourced but UNCONFIRMED" for the
full sourcing and reasoning.

**GPU is off** -- MiniReconModel is ~3M params, inference-only (no
backward pass), CPU should be tractable for ~1541 forward passes.

## 0. Setup: clone repo, install deps

In [ ]:
REPO_URL = "https://github.com/ritiksharma3/endoslam.git"

!git clone $REPO_URL repo
%cd repo
!pip install -q -r environment/requirements.txt  # includes open3d, pandas, openpyxl, scipy -- no torch reinstall needed (GPU off)

import sys
sys.path.insert(0, ".")


## 1. Resolve dataset root + Phase 3 checkpoint

In [ ]:
import os

def find_endoslam_root(base="/kaggle/input", max_depth=4):
    for root, dirs, _files in os.walk(base):
        depth = root[len(base):].count(os.sep)
        if depth > max_depth:
            dirs[:] = []
            continue
        if os.path.basename(root).lower() == "endoslam":
            return root
    return None

def find_phase3_checkpoint(base="/kaggle/input"):
    # kernel_sources mounts the phase3 training kernel's output under
    # /kaggle/input/<slug>/... -- exact path not previously exercised in
    # this repo, so glob defensively (mirrors find_endoslam_root's pattern)
    # rather than hardcoding.
    candidates = [os.path.join(r, f) for r, _, fs in os.walk(base) for f in fs
                  if f.startswith("epoch_") and f.endswith(".pt")]
    if not candidates:
        return None
    return max(candidates, key=lambda p: int(os.path.basename(p).split("_")[1].split(".")[0]))

DATA_ROOT = find_endoslam_root()
assert DATA_ROOT, "could not find an endoslam dir under /kaggle/input"
print("DATA_ROOT:", DATA_ROOT)

CHECKPOINT_PATH = find_phase3_checkpoint()
assert CHECKPOINT_PATH, "could not find a Phase 3 checkpoint under /kaggle/input -- check kernel_sources mounted correctly"
print("CHECKPOINT_PATH:", CHECKPOINT_PATH)


## 2. Load config, dataset (full sequence), and the trained model

In [ ]:
import yaml
import torch

with open("configs/config.yaml") as f:
    config = yaml.safe_load(f)
config["data"]["root"] = DATA_ROOT

from src.data.endoslam_dataset import EndoSLAMStomachDataset

dataset = EndoSLAMStomachDataset(config, split="all", cameras=["UnityCam"],
                                  context_window=config["reconstruction"]["context_window"])
print(f"full UnityCam sequence: {len(dataset)} windows (== frame count - context_window + 1)")

from src.reconstruction.model import MiniReconModel

model = MiniReconModel(pretrained=False, depth_head_channels=config["reconstruction"]["depth_head_channels"])
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"loaded checkpoint: epoch={checkpoint['epoch']}, global_step={checkpoint['global_step']}, "
      f"val_depth_absrel={checkpoint.get('val_depth_absrel')}")


## 3. Stage 1 - pose-chain math unit check (pure math, no data)

In [ ]:
import torch as _torch
from src.reconstruction.geometry import absolute_poses_from_relative, relative_pose_from_absolute, rotation_6d_to_matrix

_torch.manual_seed(0)
N = 10
_r = rotation_6d_to_matrix(_torch.randn(N + 1, 6))
_t = _torch.randn(N + 1, 3) * 0.1
_abs = _torch.eye(4).unsqueeze(0).repeat(N + 1, 1, 1)
_abs[:, :3, :3] = _r
_abs[:, :3, 3] = _t
_rel = relative_pose_from_absolute(_abs[:-1], _abs[1:])
_recovered = absolute_poses_from_relative(_abs[0], _rel[:, :3, :3], _rel[:, :3, 3])
assert _torch.allclose(_recovered, _abs, atol=1e-4), "pose-chain round-trip failed"
print("pose-chain math unit check: PASSED")


## 4. Stage 2 - GT-mode reconstruction, sweep y_down

Backprojects this dataset's own GT depth through its own GT absolute pose
(no model, no chaining -- every frame already has a valid absolute pose).
This is the empirical gate: does either y_down setting produce a coherent
tube shape? Renders matplotlib scatter previews (headless-safe, no OpenGL/
display dependency) rather than Open3D's interactive viewer, since this
runs on a headless Kaggle container.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import open3d as o3d

from src.fusion.reconstruct import reconstruct_gt

os.makedirs("/kaggle/working/pointclouds", exist_ok=True)
os.makedirs("/kaggle/working/previews", exist_ok=True)

def save_preview(pcd, name, max_points=20000):
    pts = np.asarray(pcd.points)
    cols = np.asarray(pcd.colors)
    print(f"{name}: {len(pts)} points, extent X=[{pts[:,0].min():.3f},{pts[:,0].max():.3f}] "
          f"Y=[{pts[:,1].min():.3f},{pts[:,1].max():.3f}] Z=[{pts[:,2].min():.3f},{pts[:,2].max():.3f}]")
    if len(pts) > max_points:
        idx = np.random.choice(len(pts), max_points, replace=False)
        pts, cols = pts[idx], cols[idx]
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    for ax, (i, j, label) in zip(axes, [(0, 2, "top (X-Z)"), (1, 2, "side (Y-Z)"), (0, 1, "front (X-Y)")]):
        ax.scatter(pts[:, i], pts[:, j], c=cols, s=0.5)
        ax.set_title(f"{name}: {label}")
        ax.set_aspect("equal")
    plt.tight_layout()
    out_path = f"/kaggle/working/previews/{name}.png"
    plt.savefig(out_path, dpi=100)
    plt.close(fig)
    print(f"saved preview: {out_path}")

for y_down in (True, False):
    name = f"gt_ydown_{y_down}"
    pcd = reconstruct_gt(dataset, config, y_down=y_down)
    o3d.io.write_point_cloud(f"/kaggle/working/pointclouds/{name}.ply", pcd)
    save_preview(pcd, name)


## 5. Stage 3 - predicted-mode reconstruction

Uses `config['fusion']['depth_axis_y_down']` (the notebook's config
default) since the visual outcome of Stage 2 isn't known until this
notebook is actually run -- re-run this cell with the other value if
Stage 2's images show the opposite convention is correct.

In [ ]:
from src.fusion.reconstruct import reconstruct_predicted

y_down = config["fusion"]["depth_axis_y_down"]
anchor_pose = dataset[0]["poses"][0].numpy()  # GT frame 0's absolute pose -- world-frame origin choice only

pcd_pred = reconstruct_predicted(dataset, model, config, y_down=y_down, anchor_pose=anchor_pose)
o3d.io.write_point_cloud("/kaggle/working/pointclouds/predicted.ply", pcd_pred)
save_preview(pcd_pred, "predicted")


## Done

Check `previews/gt_ydown_True.png` and `previews/gt_ydown_False.png` --
whichever looks like a coherent tube/lumen shape (not a scattered mess)
confirms the FOV/near-far/axis-convention hypothesis (or refutes both,
meaning the camera-model assumptions need revisiting -- see PROGRESS.md).
If one clearly works, set `fusion.depth_axis_y_down` in `configs/
config.yaml` accordingly and re-run cell 5 if it used the wrong one.

`previews/predicted.png` and `pointclouds/predicted.ply` are only a
meaningful Phase 3 quality signal once a GT-mode preview above looks
right -- otherwise a bad predicted-mode result could just mean the
pipeline geometry is wrong, not that the model is bad.

Download `pointclouds/*.ply` for local interactive viewing (Open3D) once
validated.